# XJTU mean and worst-error audit

This notebook tests a causal correction for current SOH and a linear model that penalizes both mean and worst RUL error. It also checks what happens when that RUL is used in the existing future SOH simulation.

Select the project Python kernel and run all cells. The notebook reads the saved reports for Notebooks 05–08 and writes CSVs, a summary, and a results ZIP under reports. It uses the same held-out batteries and includes one row per battery in the comparison. The results describe the existing XJTU cohort.

In [ ]:
from collections import defaultdict, deque
from contextlib import contextmanager
import csv
from io import TextIOWrapper
import json
from pathlib import Path, PurePosixPath
from statistics import mean, median
from zipfile import ZIP_DEFLATED, ZipFile
import warnings

import numpy as np
from scipy.optimize import OptimizeWarning, linprog
from threadpoolctl import threadpool_limits

OUTPUT_ROOT = None  # Optional parent folder for the results.
NB05 = "lifeguard_soh_error_audit_results"
NB06 = "lifeguard_rul_charge_features_results"
NB07 = "lifeguard_digital_twin_validation_results"
NB08 = "lifeguard_fixed_anchor_soh_results"
OUTPUT_NAME = "lifeguard_average_and_worst_error_results"

FIRST_CYCLE = "first_cycle_calibrated"
WITH_HISTORY = "first_cycle_and_history_ridge"
SOH_MODELS = (FIRST_CYCLE, WITH_HISTORY)
HISTORY_START = 25
HISTORY_SIZE = 15
GUARD_START = 100
RISE_ALLOWANCE_PP = 0.5

ANCHOR = 100
EOL_SOH = 80.0
ALPHAS = (0.1, 1.0, 5.0)
TRAIN_WORST_WEIGHT = 0.5
TUNE_WORST_WEIGHT = 0.25
FIELDS = ("recent_charge_ah", "recent_4_0_to_4_1_s", "recent_taper_s")
POOLED = "ridge_charge_history_pooled"
RECENT = "ridge_3c_recent_signals"
ENSEMBLE = "ridge_average"
ROBUST = "tail_penalized_linear"
INCUMBENT = "late_life_calibrated"

cwd = Path.cwd()
project = next((path for path in (cwd, *cwd.parents)
                if (path / "README_LifeGuard_RL.md").is_file() and (path / "reports").is_dir()), None)
if project is None:
    raise FileNotFoundError("Open the notebook from the LifeGuard-RL project.")
reports = project / "reports"
output = (Path(OUTPUT_ROOT).expanduser().resolve() if OUTPUT_ROOT is not None else reports) / OUTPUT_NAME


@contextmanager
def read_report(report, name):
    folder = reports / report
    if (folder / name).is_file():
        with (folder / name).open(encoding="utf-8", newline="") as file:
            yield csv.DictReader(file)
        return
    with ZipFile(folder.with_suffix(".zip")) as archive:
        matches = [member for member in archive.namelist() if PurePosixPath(member).name == name]
        if len(matches) != 1:
            raise ValueError(f"Expected one {name} in {report}.zip")
        with TextIOWrapper(archive.open(matches[0]), encoding="utf-8", newline="") as file:
            yield csv.DictReader(file)


def average(data):
    return mean(data) if data else None


def metrics(data):
    return average(data), max(data, default=None), float(np.quantile(data, 0.95)) if data else None

In [ ]:
def guarded_predictions(trace):
    history = deque(maxlen=HISTORY_SIZE)
    lowest_median = float("inf")
    previous_cycle = 0
    for row in sorted(trace, key=lambda row: int(row["cycle"])):
        cycle = int(row["cycle"])
        if cycle <= previous_cycle:
            raise ValueError("Expected one SOH estimate per cell and cycle.")
        previous_cycle = cycle
        original = float(row["predicted_soh_pct"])
        current = row["status"] == "current"
        if current and cycle > GUARD_START:
            value = min(original, lowest_median + RISE_ALLOWANCE_PP)
        else:
            value = original
        yield row, value

        # Only this and later cycles can use the newly available reading.
        if current and cycle >= HISTORY_START:
            history.append(original)
            if len(history) == HISTORY_SIZE:
                lowest_median = min(lowest_median, median(history))


traces = defaultdict(list)
with read_report(NB05, "predictions.csv") as rows:
    for row in rows:
        if row["model"] in SOH_MODELS:
            traces[row["design"], row["model"], row["cell_id"]].append(row)

expected = {}
with read_report(NB05, "metrics.csv") as rows:
    for row in rows:
        if row["model"] in SOH_MODELS:
            expected[row["design"], row["model"]] = row

soh_cycle_rows, soh_cell_rows = [], []
soh_scores = defaultdict(list)
for (design, model, cell), trace in sorted(traces.items()):
    valid = [row for row in trace if row["predicted_soh_pct"] and row["observed_soh_pct"]]
    scored = []
    for row, guarded in guarded_predictions(valid):
        actual = float(row["observed_soh_pct"])
        original = float(row["predicted_soh_pct"])
        detail = {
            "design": design, "model": model, "cell_id": cell, "fold": row["fold"],
            "cycle": int(row["cycle"]), "status": row["status"], "observed_soh_pct": actual,
            "baseline_soh_pct": original, "guarded_soh_pct": guarded,
            "baseline_abs_error_pp": abs(original - actual),
            "guarded_abs_error_pp": abs(guarded - actual),
        }
        scored.append(detail)
        soh_cycle_rows.append(detail)
    for label in ("baseline", "guarded"):
        errors = [row[f"{label}_abs_error_pp"] for row in scored]
        near = [row[f"{label}_abs_error_pp"] for row in scored if row["observed_soh_pct"] <= 82]
        mae, worst, p95 = metrics(errors)
        soh_cell_rows.append({
            "design": design, "model": model, "variant": label, "cell_id": cell,
            "fold": trace[0]["fold"], "recorded_cycles": len(trace), "paired_cycles": len(errors),
            "mae_pp": mae, "worst_error_pp": worst, "p95_error_pp": p95,
            "near_eol_cycles": len(near), "near_eol_mae_pp": average(near),
        })
    soh_scores[design, model].extend(scored)

soh_metrics = []
for (design, model), scores in sorted(soh_scores.items()):
    for variant in ("baseline", "guarded"):
        group = [row for row in soh_cell_rows if (row["design"], row["model"], row["variant"]) == (design, model, variant)]
        all_errors = [row[f"{variant}_abs_error_pp"] for row in scores]
        worst_row = max(scores, key=lambda row: row[f"{variant}_abs_error_pp"])
        count = sum(row["recorded_cycles"] for row in group)
        result = {
            "design": design, "model": model, "variant": variant, "cells": len(group),
            "paired_cycles": len(all_errors), "coverage_pct": 100 * len(all_errors) / count,
            "mae_pp": average([row["mae_pp"] for row in group if row["mae_pp"] is not None]),
            "worst_error_pp": max(all_errors), "p95_error_pp": float(np.quantile(all_errors, 0.95)),
            "near_eol_mae_pp": average([row["near_eol_mae_pp"] for row in group if row["near_eol_mae_pp"] is not None]),
            "worst_cell": worst_row["cell_id"], "worst_cycle": worst_row["cycle"],
        }
        if variant == "baseline":
            saved = expected[design, model]
            for key, source in (("mae_pp", "paired_macro_mae_pp"), ("worst_error_pp", "paired_worst_error_pp"),
                                ("near_eol_mae_pp", "paired_near_eol_mae_pp")):
                if not np.isclose(result[key], float(saved[source]), rtol=0, atol=1e-9):
                    raise ValueError(f"Notebook 05 {source} differs for {design}, {model}")
        soh_metrics.append(result)
print(f"Scored {len(soh_cycle_rows):,} paired SOH rows.")

In [ ]:
def at_anchor(row, rate):
    return int(row["landmark_cycle"]) == ANCHOR and int(row["charge_rate_c"]) == rate


def load_rul_inputs():
    with read_report(NB06, "landmark_features.csv") as rows:
        cohort = {row["cell_id"]: row for row in rows if at_anchor(row, 3)}
    cells = sorted(cohort)
    values = np.array([[float(cohort[cell][field]) for field in FIELDS] for cell in cells])
    truth = np.array([float(cohort[cell]["observed_rul_cycles"]) for cell in cells])
    with read_report(NB06, "cell_splits.csv") as rows:
        splits = {(int(row["fold"]), row["role"], row["cell_id"]) for row in rows if at_anchor(row, 3)}
    with read_report(NB07, "cell_splits.csv") as rows:
        reference = {(int(row["fold"]), row["role"], row["cell_id"]) for row in rows if row["charge_rate_c"] == "3"}
    if splits != reference:
        raise ValueError("Notebook 06 and 07 folds differ.")
    test_folds = {cell: fold for fold, role, cell in splits if role == "test"}
    if len(set(test_folds.values())) != 5 or set(test_folds) != set(cohort):
        raise ValueError("Expected the five held-out folds for the 3C cohort.")
    for fold in set(test_folds.values()):
        if {cell for cell in cells if test_folds[cell] != fold} != {
            cell for f, role, cell in splits if f == fold and role == "train"
        }:
            raise ValueError(f"Inconsistent training membership in fold {fold}.")
    saved = defaultdict(dict)
    with read_report(NB06, "rul_predictions.csv") as rows:
        for row in rows:
            rate = int(row["charge_rate_c"])
            model = row["model"]
            if int(row["landmark_cycle"]) != ANCHOR:
                continue
            if rate == 3 and model in (POOLED, RECENT):
                cell = row["cell_id"]
                if int(row["fold"]) != test_folds[cell] or float(row["observed_rul_cycles"]) != float(cohort[cell]["observed_rul_cycles"]):
                    raise ValueError(f"Reference prediction does not match the test split for {cell}.")
                saved[model][cell] = float(row["predicted_rul_cycles"])
            elif rate == 2 and model == "protocol_median":
                saved[model][row["cell_id"]] = row
    if not np.isfinite(values).all() or not np.isfinite(truth).all():
        raise ValueError("The cycle-100 RUL feature table must be finite.")
    return cells, values, truth, test_folds, saved


def predict_tail_aware(train_x, train_y, test_x, alpha):
    center = train_x.mean(axis=0)
    scale = train_x.std(axis=0)
    scale[scale == 0] = 1
    x = (train_x - center) / scale
    z = (test_x - center) / scale
    n, k = x.shape

    # Variables are intercept, slopes, per-cell absolute errors, max error, and absolute slopes.
    design = np.c_[np.ones(n), x]
    error_identity = np.eye(n)
    empty_error = np.zeros((n, k + 1))
    slope = np.c_[np.zeros(k), np.eye(k)]
    empty_slope = np.zeros((k, n + 1))
    constraints = np.r_[
        np.c_[design, -error_identity, empty_error],
        np.c_[-design, -error_identity, empty_error],
        np.c_[empty_error, error_identity, -np.ones(n), np.zeros((n, k))],
        np.c_[slope, empty_slope, -np.eye(k)],
        np.c_[-slope, empty_slope, -np.eye(k)],
    ]
    upper = np.r_[train_y, -train_y, np.zeros(n + 2 * k)]
    cost = np.r_[np.zeros(k + 1), np.full(n, 1 / n), TRAIN_WORST_WEIGHT, np.full(k, alpha)]
    with warnings.catch_warnings():
        # Forward the single-thread setting to HiGHS.
        warnings.filterwarnings("ignore", category=OptimizeWarning, message="Unrecognized options detected.*")
        fit = linprog(cost, A_ub=constraints, b_ub=upper, method="highs",
                      bounds=[(None, None)] * (k + 1) + [(0, None)] * (n + 1 + k), options={"threads": 1})
    if not fit.success:
        raise RuntimeError(f"Linear model could not be fitted: {fit.message}")
    return np.maximum(fit.x[0] + z @ fit.x[1:k + 1], 0)


def nested_predictions(values, truth, folds):
    predictions = np.full(len(truth), np.nan)
    tuning = []
    for fold in sorted(set(folds)):
        fitting = np.flatnonzero(folds != fold)
        testing = np.flatnonzero(folds == fold)

        def inner_score(alpha):
            errors = []
            for held in fitting:
                train = fitting[fitting != held]
                estimate = predict_tail_aware(values[train], truth[train], values[[held]], alpha)[0]
                errors.append(abs(float(estimate - truth[held])))
            return mean(errors) + TUNE_WORST_WEIGHT * max(errors)

        score, alpha = min((inner_score(alpha), alpha) for alpha in ALPHAS)
        predictions[testing] = predict_tail_aware(values[fitting], truth[fitting], values[testing], alpha)
        tuning.append({"fold": fold, "alpha": alpha, "inner_score_cycles": score,
                       "train_cells": len(fitting), "test_cells": len(testing)})
    if not np.isfinite(predictions).all():
        raise ValueError("Missing held-out RUL predictions.")
    return predictions, tuning


cells, values, truth, test_folds, saved = load_rul_inputs()
folds = np.array([test_folds[cell] for cell in cells])
with threadpool_limits(limits=1):
    robust, tuning = nested_predictions(values, truth, folds)
direct = {
    POOLED: np.array([saved[POOLED][cell] for cell in cells]),
    RECENT: np.array([saved[RECENT][cell] for cell in cells]),
}
direct[ENSEMBLE] = (direct[POOLED] + direct[RECENT]) / 2
direct[ROBUST] = robust
rul_cells = []
for name, predictions in direct.items():
    for cell, fold, actual, prediction in zip(cells, folds, truth, predictions):
        rul_cells.append({
            "charge_rate_c": 3, "landmark_cycle": ANCHOR, "model": name, "cell_id": cell, "fold": int(fold),
            "observed_eol_cycle": int(ANCHOR + actual), "observed_rul_cycles": int(actual),
            "predicted_rul_cycles": float(prediction), "signed_error_cycles": float(prediction - actual),
            "absolute_error_cycles": abs(float(prediction - actual)),
        })
for cell, row in sorted(saved["protocol_median"].items()):
    actual, prediction = map(float, (row["observed_rul_cycles"], row["predicted_rul_cycles"]))
    rul_cells.append({
        "charge_rate_c": 2, "landmark_cycle": ANCHOR, "model": "protocol_median", "cell_id": cell,
        "fold": int(row["fold"]), "observed_eol_cycle": int(ANCHOR + actual), "observed_rul_cycles": int(actual),
        "predicted_rul_cycles": prediction, "signed_error_cycles": prediction - actual,
        "absolute_error_cycles": abs(prediction - actual),
    })
rul_metrics = []
for (rate, model) in dict.fromkeys((row["charge_rate_c"], row["model"]) for row in rul_cells):
    group = [r for r in rul_cells if (r["charge_rate_c"], r["model"]) == (rate, model)]
    mae, worst, p95 = metrics([row["absolute_error_cycles"] for row in group])
    short = [row["absolute_error_cycles"] for row in group if row["observed_eol_cycle"] <= 200]
    rul_metrics.append({
        "charge_rate_c": rate, "landmark_cycle": ANCHOR, "model": model, "cells": len(group),
        "mae_cycles": mae, "worst_error_cycles": worst, "p95_error_cycles": p95,
        "short_life_cells": len(short), "short_life_mae_cycles": average(short),
        "worst_cell": max(group, key=lambda row: row["absolute_error_cycles"])["cell_id"],
    })

In [ ]:
def reference_trajectories(report, model):
    observed, forecasts = defaultdict(dict), defaultdict(dict)
    with read_report(report, "soh_trajectories.csv") as rows:
        for row in rows:
            if at_anchor(row, 3) and row["initializer"] == "charge" and row["model"] == model:
                cell, cycle = row["cell_id"], int(row["cycle"])
                observed[cell][cycle] = float(row["observed_soh_pct"])
                forecasts[cell][cycle] = float(row["predicted_soh_pct"])
    return observed, forecasts


def project_soh(original, original_rul, new_rul, template, phase):
    duration = (original_rul + new_rul) / 2
    gate = min(50, int(original_rul) // 2, int(duration // 2))
    damage_at_gate = np.interp(gate / duration, phase, template)
    curve = original.copy()
    steps = np.arange(gate + 1, len(original))
    damage = np.interp(steps / duration, phase, template)
    remaining = (1 - damage) / (1 - damage_at_gate)
    curve[steps] = np.where(steps >= duration, EOL_SOH, EOL_SOH + (original[gate] - EOL_SOH) * remaining)
    return curve


def compare_future_soh():
    observed, baseline = reference_trajectories(NB07, "soh_dependent")
    aligned_observed, incumbent = reference_trajectories(NB08, INCUMBENT)
    if observed != aligned_observed:
        raise ValueError("Notebook 07 and 08 observed SOH timelines differ.")
    ends = {cell: ANCHOR + int(t) for cell, t in zip(cells, truth)}
    for cell in cells:
        if set(observed[cell]) != set(range(ANCHOR, ends[cell] + 1)):
            raise ValueError(f"Missing evaluation cycles for {cell}.")
    with read_report(NB07, "rollout_starts.csv") as rows:
        old_rul = {row["cell_id"]: int(float(row["predicted_rul_cycles"])) for row in rows
                   if at_anchor(row, 3) and row["initializer"] == "charge" and row["model"] == "soh_dependent"}
    calibration = defaultdict(list)
    with read_report(NB07, "transition_calibration.csv") as rows:
        for row in rows:
            if row["charge_rate_c"] == "3" and row["used"] == "True":
                calibration[int(row["fold"])].append(
                    (float(row["bin_bottom_soh_pct"]) + 1.0, float(row["model_fade_pp_per_cycle"]))
                )

    phase = np.linspace(0, 1, 101)
    templates = {}
    for fold in sorted(calibration):
        shapes = []
        for cell in cells:
            if test_folds[cell] == fold:
                continue
            ages = np.arange(ANCHOR, ends[cell] + 1)
            levels = np.array([observed[cell][t] for t in ages])
            if levels[0] <= EOL_SOH:
                continue
            aligned = np.interp(ANCHOR + phase * (ends[cell] - ANCHOR), ages, levels)
            damage = np.clip((levels[0] - aligned) / (levels[0] - EOL_SOH), 0, 1 - 1e-9)
            damage = np.maximum.accumulate(damage)
            damage[-1] = 1.0
            shapes.append(damage)
        templates[fold] = np.median(shapes, axis=0)

    rows = []
    for i, cell in enumerate(cells):
        fold = test_folds[cell]
        x, decay = zip(*sorted(calibration[fold]))
        original = [baseline[cell][ANCHOR]]
        for _ in range(600):
            current = original[-1]
            original.append(max(EOL_SOH, current - np.interp(current, x, decay)))
        original = np.asarray(original)
        cycles = np.arange(ANCHOR, ends[cell] + 1)
        indices = cycles - ANCHOR
        rebuilt = project_soh(original, old_rul[cell], direct[POOLED][i], templates[fold], phase)
        if not np.allclose(rebuilt[indices], [incumbent[cell][t] for t in cycles], rtol=0, atol=1e-9):
            raise ValueError(f"Notebook 08 curve could not be reproduced for {cell}.")
        proposed = project_soh(original, old_rul[cell], direct[ROBUST][i], templates[fold], phase)
        for name, series in ((INCUMBENT, rebuilt), (ROBUST, proposed)):
            eol = np.flatnonzero(series <= EOL_SOH)
            errors = np.abs(series[indices] - np.asarray([observed[cell][t] for t in cycles]))
            rows.append({
                "cell_id": cell, "fold": fold, "model": name,
                "full_curve_mae_pp": float(errors[1:].mean()),
                "last_20_mae_pp": float(errors[-20:].mean()),
                "last_20_worst_error_pp": float(errors[-20:].max()),
                "horizon_10_error_pp": float(errors[10]) if len(errors) > 10 else None,
                "horizon_50_error_pp": float(errors[50]) if len(errors) > 50 else None,
                "eol_error_cycles": abs(int(eol[0]) - int(truth[i])) if eol.size else None,
            })
    return rows


future_cells = compare_future_soh()
future_metrics = []
for name in (INCUMBENT, ROBUST):
    group = [r for r in future_cells if r["model"] == name]
    future_metrics.append({
        "charge_rate_c": 3, "landmark_cycle": ANCHOR, "model": name, "cells": len(group),
        "full_curve_mae_pp": mean(r["full_curve_mae_pp"] for r in group),
        "last_20_mae_pp": mean(r["last_20_mae_pp"] for r in group),
        "last_20_worst_error_pp": max(r["last_20_worst_error_pp"] for r in group),
        "horizon_10_mae_pp": average([r["horizon_10_error_pp"] for r in group if r["horizon_10_error_pp"] is not None]),
        "horizon_50_mae_pp": average([r["horizon_50_error_pp"] for r in group if r["horizon_50_error_pp"] is not None]),
        "eol_predictions": sum(r["eol_error_cycles"] is not None for r in group),
        "eol_mae_cycles": average([r["eol_error_cycles"] for r in group if r["eol_error_cycles"] is not None]),
    })

In [ ]:
def fmt(value):
    return "—" if value is None else f"{value:.3f}"


def table_row(label, metrics):
    return "| " + " | ".join([label, *map(fmt, metrics)]) + " |"


def make_summary():
    texts = [
        "# XJTU mean and worst-error audit",
        "",
        "## Current SOH",
        "",
        "SOH errors are in percentage points. MAE weights each held-out cell equally. "
        "Near-EOL scores use measured SOH of 82% or lower. The guard uses past valid charge-based estimates.",
        "",
        "| Evaluation | Source model | Variant | MAE | Worst | Near-EOL MAE | Coverage (%) |",
        "| --- | --- | --- | ---: | ---: | ---: | ---: |",
    ]
    names = {FIRST_CYCLE: "First-cycle correction", WITH_HISTORY: "First-cycle + history"}
    for row in soh_metrics:
        label = " | ".join([row["design"], names[row["model"]], row["variant"]])
        texts.append(table_row(label, [row[key] for key in ("mae_pp", "worst_error_pp", "near_eol_mae_pp", "coverage_pct")]))
    texts.extend([
        "",
        "## RUL at cycle 100",
        "",
        "MAE and maximum errors are in cycles. The short-life group reached EOL by cycle 200.",
        "",
        "| Charge rate | Model | MAE | Worst | Short-life MAE |",
        "| --- | --- | ---: | ---: | ---: |",
    ])
    for row in rul_metrics:
        texts.append(table_row(f"{row['charge_rate_c']}C | {row['model']}",
                               [row[key] for key in ("mae_cycles", "worst_error_cycles", "short_life_mae_cycles")]))
    texts.extend([
        "",
        "## 3C future SOH from cycle 100",
        "",
        "| RUL input | Full curve MAE (pp) | Last 20 MAE (pp) | Last 20 worst (pp) | +50 MAE (pp) | EOL timing MAE (cycles) |",
        "| --- | ---: | ---: | ---: | ---: | ---: |",
    ])
    for row in future_metrics:
        texts.append(table_row(row["model"], [row[key] for key in ("full_curve_mae_pp", "last_20_mae_pp",
                                                                  "last_20_worst_error_pp", "horizon_50_mae_pp", "eol_mae_cycles")]))
    texts.extend([
        "",
        "The RUL model tunes the coefficient penalty inside each outer training fold. "
        "The guard and loss weights were chosen during exploration on this XJTU cohort. "
        "The results describe held-out cells within that cohort.",
    ])
    return "\n".join(texts) + "\n"


tables = {
    "soh_cycle_comparison.csv": soh_cycle_rows,
    "soh_cell_metrics.csv": soh_cell_rows,
    "soh_metrics.csv": soh_metrics,
    "rul_cell_predictions.csv": rul_cells,
    "rul_metrics.csv": rul_metrics,
    "rul_fold_tuning.csv": tuning,
    "future_soh_cell_metrics.csv": future_cells,
    "future_soh_metrics.csv": future_metrics,
}
output.mkdir(parents=True, exist_ok=True)
for name, rows in tables.items():
    with (output / name).open("w", encoding="utf-8", newline="") as file:
        writer = csv.DictWriter(file, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)
summary = make_summary()
(output / "summary.md").write_text(summary, encoding="utf-8")
config = {
    "guard_start": GUARD_START, "history_start": HISTORY_START, "history_size": HISTORY_SIZE,
    "rise_allowance_pp": RISE_ALLOWANCE_PP, "landmark_cycle": ANCHOR, "eol_soh_pct": EOL_SOH,
    "features": FIELDS, "alphas": ALPHAS, "train_worst_weight": TRAIN_WORST_WEIGHT,
    "tune_worst_weight": TUNE_WORST_WEIGHT, "source_reports": (NB05, NB06, NB07, NB08),
}
(output / "run_config.json").write_text(json.dumps(config, indent=2) + "\n", encoding="utf-8")
archive = output.with_suffix(".zip")
with ZipFile(archive, "w", compression=ZIP_DEFLATED) as bundle:
    for name in (*tables, "summary.md", "run_config.json"):
        bundle.write(output / name, name)

from IPython import get_ipython
from IPython.display import Markdown, display

if get_ipython() is not None:
    display(Markdown(summary))
else:
    print(summary)
print(f"Saved {archive}")